# AMI Meeting Corpus: Transcript Reconstruction & Dataset Construction

Builds a clean, structured dataset from the AMI manual annotations:

1. Download and inspect the corpus
2. Validate word/segment alignment
3. Reconstruct speaker-labelled transcripts
4. Clean transcripts
5. Pair transcripts with abstractive annotations and export the dataset

## 1. Download and inspect the corpus

In [ ]:
!pip install -q pandas

In [ ]:
# DATASET DOWNLOAD

!wget -q -O ami_manual_annotations.zip \
  https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_manual_1.6.2.zip

!unzip -q ami_manual_annotations.zip -d ami_corpus

print("\n\n\nAMI manual annotations downloaded and extracted!")

In [ ]:
# CHECKING FILE STRUCTURE

import os

for root, dirs, files in os.walk("ami_corpus"):
    for file in files[:]:
        print(os.path.join(root, file))

In [ ]:
from pathlib import Path
import xml.dom.minidom

base = Path("ami_corpus")

# Read the dataset documentation
readme = base / "00README_MANUAL.txt"

print("===== AMI README =====")
print(readme.read_text(errors="replace")[:5000])

# Inspect representative annotation files
folders = [
    "abstractive",
    "decision/manual",
    "participantSummaries"
]

for folder in folders:
    files = list((base / folder).glob("*.xml"))

    print(f"\n===== {folder} =====")
    print("Available XML files:", len(files))

    if files:
        sample_file = files[0]
        print("Sample:", sample_file.name)

        xml_text = sample_file.read_text(errors="replace")
        formatted = xml.dom.minidom.parseString(xml_text).toprettyxml()

        print(formatted[:6000])

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

base = Path("ami_corpus/abstractive")

section_counts = Counter()
sentence_totals = Counter()
empty_sections = Counter()

files = sorted(base.glob("*.xml"))

for file in files:
    root = ET.parse(file).getroot()

    for section in ["abstract", "actions", "decisions", "problems"]:
        elements = [
            el for el in root.iter()
            if el.tag.split("}")[-1] == section
        ]

        if elements:
            section_counts[section] += 1

            sentence_count = sum(
                1 for el in elements
                for child in el.iter()
                if child.tag.split("}")[-1] == "sentence"
            )

            sentence_totals[section] += sentence_count

            if sentence_count == 0:
                empty_sections[section] += 1

print("Total abstractive files:", len(files))

print("\nSECTION COVERAGE")
for section in ["abstract", "actions", "decisions", "problems"]:
    print(
        f"{section:12} | "
        f"Files: {section_counts[section]:3} | "
        f"Sentences: {sentence_totals[section]:5} | "
        f"Empty: {empty_sections[section]}"
    )

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

base = Path("ami_corpus")

# Select one meeting from the abstractive annotations
annotation_file = sorted((base / "abstractive").glob("*.xml"))[0]
meeting_id = annotation_file.name.replace(".abssumm.xml", "")

print("Meeting ID:", meeting_id)

# Find its participant word files
word_files = sorted((base / "words").glob(f"{meeting_id}.*.words.xml"))

print("\nMatching transcript files:", len(word_files))

for file in word_files:
    print("\n" + "=" * 60)
    print("File:", file.name)

    root = ET.parse(file).getroot()

    # Display the XML structure and a sample of its contents
    print("Root tag:", root.tag)
    print("Root attributes:", root.attrib)

    for element in list(root)[:8]:
        print(
            "Tag:", element.tag,
            "| Attributes:", element.attrib,
            "| Text:", (element.text or "").strip()[:100]
        )

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

base = Path("ami_corpus")
meeting_id = "ES2002a"

events = []
non_word_elements = {}

for file in sorted((base / "words").glob(f"{meeting_id}.*.words.xml")):
    speaker = file.name.split(".")[1]

    root = ET.parse(file).getroot()

    for element in root:
        tag = element.tag.split("}")[-1]

        if tag != "w":
            non_word_elements[tag] = non_word_elements.get(tag, 0) + 1
            continue

        text = (element.text or "").strip()

        events.append({
            "speaker": speaker,
            "text": text,
            "start": float(element.attrib["starttime"]),
            "end": float(element.attrib["endtime"]),
            "punctuation": element.attrib.get("punc") == "true"
        })

# Merge participant streams by timestamp
events.sort(key=lambda x: (x["start"], x["end"]))

print("Total word/punctuation elements:", len(events))
print("Other element types:", non_word_elements)

print("\nFIRST 100 CHRONOLOGICAL EVENTS")
for event in events[:100]:
    print(
        f'{event["start"]:8.2f} | '
        f'{event["speaker"]} | '
        f'{event["text"]}'
    )

## 2. Corpus audit and validation of word/segment alignment

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

base = Path("ami_corpus")
annotation_dir = base / "abstractive"
words_dir = base / "words"

results = []
timing_issues = []

annotation_files = sorted(annotation_dir.glob("*.abssumm.xml"))

for annotation_file in annotation_files:
    meeting_id = annotation_file.name.replace(".abssumm.xml", "")
    word_files = sorted(words_dir.glob(f"{meeting_id}.*.words.xml"))

    events = []
    all_text = []
    non_word_counts = {}
    punctuation_count = 0
    missing_start = 0
    missing_end = 0
    invalid_timing = 0

    for file in word_files:
        speaker = file.name.split(".")[1]
        root = ET.parse(file).getroot()

        for element in root:
            tag = element.tag.split("}")[-1]

            if tag != "w":
                non_word_counts[tag] = non_word_counts.get(tag, 0) + 1
                continue

            text = (element.text or "").strip()

            if not text:
                continue

            all_text.append(text)

            if element.attrib.get("punc") == "true":
                punctuation_count += 1

            start_raw = element.attrib.get("starttime")
            end_raw = element.attrib.get("endtime")

            if start_raw is None:
                missing_start += 1

            if end_raw is None:
                missing_end += 1

            try:
                if start_raw is None or end_raw is None:
                    continue

                start = float(start_raw)
                end = float(end_raw)

                if end < start:
                    invalid_timing += 1
                    continue

            except (ValueError, TypeError):
                invalid_timing += 1
                continue

            events.append({
                "speaker": speaker,
                "text": text,
                "start": start,
                "end": end
            })

    events.sort(key=lambda x: (x["start"], x["end"]))

    speaker_switches = sum(
        events[i]["speaker"] != events[i - 1]["speaker"]
        for i in range(1, len(events))
    )

    # Count overlapping timed word pairs from different speakers.
    overlapping_pairs = 0
    active = []

    for event in events:
        active = [
            previous for previous in active
            if previous["end"] > event["start"]
        ]

        if event["end"] > event["start"]:
            for previous in active:
                if previous["speaker"] != event["speaker"]:
                    overlapping_pairs += 1

            active.append(event)

    results.append({
        "meeting": meeting_id,
        "word_files": len(word_files),
        "speakers": len(set(e["speaker"] for e in events)),
        "words": len(all_text),
        "timed_words": len(events),
        "speaker_switches": speaker_switches,
        "overlapping_pairs": overlapping_pairs,
        "punctuation": punctuation_count,
        "missing_start": missing_start,
        "missing_end": missing_end,
        "invalid_timing": invalid_timing,
        "other_annotations": sum(non_word_counts.values())
    })

    if missing_start or missing_end or invalid_timing:
        timing_issues.append({
            "meeting": meeting_id,
            "missing_start": missing_start,
            "missing_end": missing_end,
            "invalid_timing": invalid_timing
        })

df = pd.DataFrame(results)

print("===== DATASET OVERVIEW =====")
print("Annotated meetings:", len(annotation_files))
print("Meetings with transcript files:", (df["word_files"] > 0).sum())
print("Meetings missing transcript files:", (df["word_files"] == 0).sum())

print("\n===== TRANSCRIPT STATISTICS =====")
print(df[["words", "speakers"]].describe().round(1))

print("\n===== SPEAKER AND TIMING ISSUES =====")
print("Meetings with speaker switches:", (df["speaker_switches"] > 0).sum())
print("Meetings with overlapping speech:", (df["overlapping_pairs"] > 0).sum())
print("Total overlapping word pairs:", df["overlapping_pairs"].sum())
print("Separate punctuation elements:", df["punctuation"].sum())
print("Other annotation elements:", df["other_annotations"].sum())

print("\n===== INCOMPLETE TIMING INFORMATION =====")
print("Meetings affected:", len(timing_issues))
print("Total missing start times:", df["missing_start"].sum())
print("Total missing end times:", df["missing_end"].sum())
print("Total invalid timing records:", df["invalid_timing"].sum())

print("\n===== LONGEST MEETINGS =====")
print(df.nlargest(10, "words")[
    ["meeting", "words", "speaker_switches", "overlapping_pairs"]
].to_string(index=False))

print("\n===== MEETINGS WITH TIMING ISSUES =====")
print(pd.DataFrame(timing_issues).to_string(index=False)
      if timing_issues else "None")

print("\n===== MEETINGS MISSING TRANSCRIPTS =====")
print(df[df["word_files"] == 0]["meeting"].tolist())

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

base = Path("ami_corpus")
meeting_id = "ES2002a"

segment_files = sorted(
    (base / "segments").glob(f"{meeting_id}.*.segments.xml")
)

print("Segment files found:", len(segment_files))

for file in segment_files[:2]:
    print(f"\n{'='*70}")
    print("FILE:", file.name)

    root = ET.parse(file).getroot()

    for element in list(root)[:15]:
        print(ET.tostring(element, encoding="unicode"))

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import pandas as pd

base = Path("ami_corpus")
meeting_id = "ES2002a"

segment_files = sorted(
    (base / "segments").glob(f"{meeting_id}.*.segments.xml")
)

results = []

for file in segment_files:
    speaker = file.name.split(".")[1]
    root = ET.parse(file).getroot()

    for segment in root:
        start = float(segment.attrib["transcriber_start"])
        end = float(segment.attrib["transcriber_end"])

        child = list(segment)[0]
        href = child.attrib["href"]

        match = re.search(
            r"words(\d+)\).*id\([^)]*words(\d+)\)",
            href
        )

        if match:
            first_word = int(match.group(1))
            last_word = int(match.group(2))
            word_count = last_word - first_word + 1
        else:
            match = re.search(r"words(\d+)\)", href)
            first_word = int(match.group(1)) if match else None
            last_word = first_word
            word_count = 1 if match else 0

        results.append({
            "speaker": speaker,
            "segment": segment.attrib.get(
                "{http://nite.sourceforge.net/}id"
            ),
            "start": start,
            "end": end,
            "duration": round(end - start, 3),
            "referenced_words": word_count
        })

df_segments = pd.DataFrame(results)

print("===== SEGMENT OVERVIEW =====")
print(
    df_segments.groupby("speaker").agg(
        segments=("segment", "count"),
        total_referenced_words=("referenced_words", "sum"),
        average_duration=("duration", "mean"),
        average_words=("referenced_words", "mean")
    ).round(2)
)

print("\n===== FIRST 10 SEGMENTS =====")
print(df_segments.head(10).to_string(index=False))

print("\n===== LONGEST SEGMENTS =====")
print(
    df_segments.nlargest(10, "duration")[
        ["speaker", "segment", "duration", "referenced_words"]
    ].to_string(index=False)
)

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import pandas as pd

base = Path("ami_corpus")
meeting_id = "ES2002a"

segments = []

for file in sorted((base / "segments").glob(f"{meeting_id}.*.segments.xml")):
    speaker = file.name.split(".")[1]

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    word_root = ET.parse(word_file).getroot()

    # Build an ID-to-word lookup
    word_lookup = {}

    for element in word_root:
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )
            if word_id:
                word_lookup[word_id] = (element.text or "").strip()

    for segment in ET.parse(file).getroot():
        start = float(segment.attrib["transcriber_start"])
        end = float(segment.attrib["transcriber_end"])

        child = list(segment)[0]
        href = child.attrib["href"]

        ids = re.findall(r"id\(([^)]+)\)", href)

        first_id = ids[0]
        last_id = ids[-1]

        first_num = int(first_id.split("words")[-1])
        last_num = int(last_id.split("words")[-1])

        words = []

        for number in range(first_num, last_num + 1):
            word_id = f"{meeting_id}.{speaker}.words{number}"

            if word_id in word_lookup:
                words.append(word_lookup[word_id])

        segments.append({
            "speaker": speaker,
            "start": start,
            "end": end,
            "text": " ".join(words)
        })

# Chronological ordering
segments.sort(key=lambda x: (x["start"], x["end"]))

# Generate transcript
transcript = "\n".join(
    f"{item['speaker']}: {item['text']}"
    for item in segments
)

print("===== SEGMENT-BASED TRANSCRIPT =====")
print(transcript[:7000])

print("\n===== STATISTICS =====")
print("Characters:", len(transcript))
print("Words:", len(transcript.split()))
print("Segments:", len(segments))

In [ ]:
# Validate Segment Coverage:

from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["A", "B", "C", "D"]:

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    word_root = ET.parse(word_file).getroot()
    segment_root = ET.parse(segment_file).getroot()

    all_word_ids = set()

    for element in word_root:
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )
            if word_id:
                all_word_ids.add(word_id)

    referenced_ids = []

    for segment in segment_root:
        for child in segment:
            href = child.attrib.get("href", "")
            ids = re.findall(r"id\(([^)]+)\)", href)

            if not ids:
                continue

            first_num = int(ids[0].split("words")[-1])
            last_num = int(ids[-1].split("words")[-1])

            for number in range(first_num, last_num + 1):
                referenced_ids.append(
                    f"{meeting_id}.{speaker}.words{number}"
                )

    unique_referenced = set(referenced_ids)

    print(f"\n===== SPEAKER {speaker} =====")
    print("Word XML elements:", len(all_word_ids))
    print("Referenced word IDs:", len(referenced_ids))
    print("Unique referenced IDs:", len(unique_referenced))
    print("Unreferenced words:", len(all_word_ids - unique_referenced))
    print("Invalid references:", len(unique_referenced - all_word_ids))
    print("Duplicate references:", len(referenced_ids) - len(unique_referenced))

In [ ]:
# Finding what the Invalid ones represent:

from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["A", "B", "C", "D"]:

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    word_root = ET.parse(word_file).getroot()
    segment_root = ET.parse(segment_file).getroot()

    all_word_ids = set()

    for element in word_root:
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )
            if word_id:
                all_word_ids.add(word_id)

    invalid_ids = set()

    for segment in segment_root:
        for child in segment:
            href = child.attrib.get("href", "")
            ids = re.findall(r"id\(([^)]+)\)", href)

            if len(ids) < 2:
                continue

            first_num = int(ids[0].split("words")[-1])
            last_num = int(ids[-1].split("words")[-1])

            for number in range(first_num, last_num + 1):
                word_id = f"{meeting_id}.{speaker}.words{number}"

                if word_id not in all_word_ids:
                    invalid_ids.add(word_id)

    print(f"\n===== SPEAKER {speaker} =====")
    print("First 20 invalid IDs:")

    for word_id in sorted(
        invalid_ids,
        key=lambda x: int(x.split("words")[-1])
    )[:20]:
        print(word_id)

In [ ]:
#Revised Diagnostic

from pathlib import Path
import xml.etree.ElementTree as ET
import re
from collections import Counter

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["A", "B", "C", "D"]:

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    word_root = ET.parse(word_file).getroot()
    segment_root = ET.parse(segment_file).getroot()

    # Map actual word IDs to their numeric suffixes
    word_ids = {}

    for element in word_root:
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )

            if word_id:
                number = int(word_id.split("words")[-1])
                word_ids[word_id] = number

    referenced_ids = []

    for segment in segment_root:
        for child in segment:
            href = child.attrib.get("href", "")
            ids = re.findall(r"id\(([^)]+)\)", href)

            if len(ids) < 2:
                continue

            first_num = int(ids[0].split("words")[-1])
            last_num = int(ids[-1].split("words")[-1])

            # Select actual words within the referenced range
            for word_id, number in word_ids.items():
                if first_num <= number <= last_num:
                    referenced_ids.append(word_id)

    counts = Counter(referenced_ids)

    unreferenced = set(word_ids) - set(counts)
    duplicated = {
        word_id: count
        for word_id, count in counts.items()
        if count > 1
    }

    print(f"\n===== SPEAKER {speaker} =====")
    print("Actual words:", len(word_ids))
    print("Referenced words:", len(referenced_ids))
    print("Unique referenced words:", len(counts))
    print("Unreferenced words:", len(unreferenced))
    print("Duplicated words:", len(duplicated))

In [ ]:
# Uncovered Word diagnostic:

from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["B", "D"]:

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    word_root = ET.parse(word_file).getroot()
    segment_root = ET.parse(segment_file).getroot()

    words = {}

    for element in word_root:
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )

            if word_id:
                words[word_id] = "".join(element.itertext()).strip()

    referenced = set()

    for segment in segment_root:
        for child in segment:
            ids = re.findall(
                r"id\(([^)]+)\)",
                child.attrib.get("href", "")
            )

            if len(ids) < 2:
                continue

            first_num = int(ids[0].split("words")[-1])
            last_num = int(ids[-1].split("words")[-1])

            for word_id in words:
                number = int(word_id.split("words")[-1])

                if first_num <= number <= last_num:
                    referenced.add(word_id)

    uncovered = set(words) - referenced

    print(f"\n===== SPEAKER {speaker} =====")

    for word_id in sorted(
        uncovered,
        key=lambda x: int(x.split("words")[-1])
    ):
        print(word_id, "->", repr(words[word_id]))

In [ ]:
# More into uncovered word placement and surroundings:

from pathlib import Path
import xml.etree.ElementTree as ET

base = Path("ami_corpus")
meeting_id = "ES2002a"

targets = {
    "B": [1024, 1509],
    "D": [881, 1154]
}

for speaker, target_numbers in targets.items():

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    root = ET.parse(word_file).getroot()

    elements = [
        element for element in root
        if element.tag.split("}")[-1] == "w"
    ]

    print(f"\n========== SPEAKER {speaker} ==========")

    for target in target_numbers:

        print(f"\n--- Around words{target} ---")

        for element in elements:
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id", ""
            )

            number = int(word_id.split("words")[-1])

            if abs(number - target) <= 3:

                text = "".join(element.itertext()).strip()

                print({
                    "id": word_id,
                    "text": text,
                    "start": element.attrib.get("starttime"),
                    "end": element.attrib.get("endtime")
                })

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")

meeting_ids = sorted(
    path.name.replace(".abssumm.xml", "")
    for path in (base / "abstractive").glob("*.abssumm.xml")
)

total_words = 0
total_covered = 0
total_single_refs = 0
total_unsegmented = 0
affected_meetings = set()
affected_speakers = 0

for meeting_id in meeting_ids:

    meeting_has_unsegmented = False

    for speaker in ["A", "B", "C", "D"]:

        word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
        segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

        if not word_file.exists() or not segment_file.exists():
            continue

        word_root = ET.parse(word_file).getroot()
        segment_root = ET.parse(segment_file).getroot()

        word_numbers = {}

        for element in word_root:
            if element.tag.split("}")[-1] == "w":
                word_id = element.attrib.get(
                    "{http://nite.sourceforge.net/}id"
                )

                if word_id:
                    word_numbers[word_id] = int(
                        word_id.split("words")[-1]
                    )

        referenced = set()
        single_refs = 0

        for segment in segment_root:
            for child in segment:
                href = child.attrib.get("href", "")
                ids = re.findall(r"id\(([^)]+)\)", href)

                if len(ids) == 1:
                    single_refs += 1
                    if ids[0] in word_numbers:
                        referenced.add(ids[0])

                elif len(ids) >= 2:
                    first_num = int(ids[0].split("words")[-1])
                    last_num = int(ids[-1].split("words")[-1])

                    for word_id, number in word_numbers.items():
                        if first_num <= number <= last_num:
                            referenced.add(word_id)

        unsegmented = set(word_numbers) - referenced

        total_words += len(word_numbers)
        total_covered += len(referenced)
        total_single_refs += single_refs
        total_unsegmented += len(unsegmented)

        if unsegmented:
            affected_speakers += 1
            meeting_has_unsegmented = True

    if meeting_has_unsegmented:
        affected_meetings.add(meeting_id)

print("Meetings:", len(meeting_ids))
print("Total words:", total_words)
print("Covered words:", total_covered)
print("Single-ID references:", total_single_refs)
print("Unsegmented words:", total_unsegmented)

percentage = (
    round(total_unsegmented / total_words * 100, 4)
    if total_words else 0
)

print("Unsegmented percentage:", percentage, "%")
print("Affected meetings:", len(affected_meetings))
print("Affected speaker files:", affected_speakers)

In [ ]:
# Locating Unsegmented words across the entire corpus:

from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")

meeting_ids = sorted(
    p.name.replace(".abssumm.xml", "")
    for p in (base / "abstractive").glob("*.abssumm.xml")
)

for meeting_id in meeting_ids:
    for speaker in ["A", "B", "C", "D"]:

        word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
        segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

        if not word_file.exists() or not segment_file.exists():
            continue

        word_root = ET.parse(word_file).getroot()
        segment_root = ET.parse(segment_file).getroot()

        words = {}

        for element in word_root:
            if element.tag.split("}")[-1] == "w":
                word_id = element.attrib.get(
                    "{http://nite.sourceforge.net/}id"
                )

                if word_id:
                    words[word_id] = {
                        "number": int(word_id.split("words")[-1]),
                        "text": "".join(element.itertext()).strip(),
                        "start": element.attrib.get("starttime"),
                        "end": element.attrib.get("endtime")
                    }

        referenced = set()

        for segment in segment_root:
            for child in segment:
                ids = re.findall(
                    r"id\(([^)]+)\)",
                    child.attrib.get("href", "")
                )

                if len(ids) == 1:
                    if ids[0] in words:
                        referenced.add(ids[0])

                elif len(ids) >= 2:
                    first = int(ids[0].split("words")[-1])
                    last = int(ids[-1].split("words")[-1])

                    for word_id, info in words.items():
                        if first <= info["number"] <= last:
                            referenced.add(word_id)

        uncovered = set(words) - referenced

        for word_id in sorted(
            uncovered,
            key=lambda x: words[x]["number"]
        ):
            info = words[word_id]

            print({
                "meeting": meeting_id,
                "speaker": speaker,
                "word_id": word_id,
                "text": info["text"],
                "start": info["start"],
                "end": info["end"]
            })

In [ ]:
'''We've validated word coverage, but not yet whether segment references preserve the original word order.

For example, we need to ensure that the words inside each segment are ordered correctly and that the segment references don't contain unexpected discontinuities.'''

from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["A", "B", "C", "D"]:

    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    root = ET.parse(segment_file).getroot()

    segment_count = 0
    invalid_ranges = []

    for segment in root:

        segment_id = segment.attrib.get(
            "{http://nite.sourceforge.net/}id", "unknown"
        )

        for child in segment:

            ids = re.findall(
                r"id\(([^)]+)\)",
                child.attrib.get("href", "")
            )

            if len(ids) >= 2:
                first = int(ids[0].split("words")[-1])
                last = int(ids[-1].split("words")[-1])

                if first > last:
                    invalid_ranges.append(
                        (segment_id, first, last)
                    )

            segment_count += 1

    print(f"\nSpeaker {speaker}")
    print("Segment references:", segment_count)
    print("Invalid reversed ranges:", len(invalid_ranges))
    print("Examples:", invalid_ranges[:5])

## 3. Transcript reconstruction

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

base = Path("ami_corpus")
meeting_id = "ES2002a"

for speaker in ["A", "B", "C", "D"]:

    word_file = base / "words" / f"{meeting_id}.{speaker}.words.xml"
    segment_file = base / "segments" / f"{meeting_id}.{speaker}.segments.xml"

    word_root = ET.parse(word_file).getroot()
    segment_root = ET.parse(segment_file).getroot()

    # Preserve the actual XML order of word elements
    word_order = {}

    for index, element in enumerate(word_root):
        if element.tag.split("}")[-1] == "w":
            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )

            if word_id:
                word_order[word_id] = index

    violations = []

    for segment in segment_root:

        segment_id = segment.attrib.get(
            "{http://nite.sourceforge.net/}id", "unknown"
        )

        for child in segment:

            ids = re.findall(
                r"id\(([^)]+)\)",
                child.attrib.get("href", "")
            )

            if len(ids) == 1:
                referenced = ids if ids[0] in word_order else []

            elif len(ids) >= 2:
                first = int(ids[0].split("words")[-1])
                last = int(ids[-1].split("words")[-1])

                referenced = [
                    word_id
                    for word_id in word_order
                    if first <= int(word_id.split("words")[-1]) <= last
                ]

            else:
                continue

            positions = [word_order[word_id] for word_id in referenced]

            if positions != sorted(positions):
                violations.append(segment_id)

    print(f"\nSpeaker {speaker}")
    print("Word-order violations:", len(violations))
    print("Examples:", violations[:5])

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re
import json

BASE = Path("ami_corpus")

def reconstruct_meeting(meeting_id):

    structured_segments = []
    unsegmented_words = []

    for speaker in ["A", "B", "C", "D"]:

        word_file = BASE / "words" / f"{meeting_id}.{speaker}.words.xml"
        segment_file = BASE / "segments" / f"{meeting_id}.{speaker}.segments.xml"

        word_root = ET.parse(word_file).getroot()
        segment_root = ET.parse(segment_file).getroot()

        # Load actual words, preserving their XML order.
        words = {}
        word_order = []

        for element in word_root:

            if element.tag.split("}")[-1] != "w":
                continue

            word_id = element.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )

            if not word_id:
                continue

            number = int(word_id.split("words")[-1])

            word = {
                "word_id": word_id,
                "number": number,
                "text": "".join(element.itertext()).strip(),
                "start": element.attrib.get("starttime"),
                "end": element.attrib.get("endtime")
            }

            words[word_id] = word
            word_order.append(word_id)

        # Track which words are assigned to segments.
        referenced_words = set()

        # Reconstruct every segment.
        for segment in segment_root:

            if segment.tag.split("}")[-1] != "segment":
                continue

            segment_id = segment.attrib.get(
                "{http://nite.sourceforge.net/}id"
            )

            segment_word_ids = []

            for child in segment:

                href = child.attrib.get("href", "")
                ids = re.findall(r"id\(([^)]+)\)", href)

                if len(ids) == 1:

                    if ids[0] in words:
                        segment_word_ids.append(ids[0])

                elif len(ids) >= 2:

                    first = int(ids[0].split("words")[-1])
                    last = int(ids[-1].split("words")[-1])

                    for word_id in word_order:

                        number = words[word_id]["number"]

                        if first <= number <= last:
                            segment_word_ids.append(word_id)

            # Remove accidental duplicates while preserving order.
            segment_word_ids = list(dict.fromkeys(segment_word_ids))

            segment_text = " ".join(
                words[word_id]["text"]
                for word_id in segment_word_ids
            )

            referenced_words.update(segment_word_ids)

            structured_segments.append({
                "meeting_id": meeting_id,
                "speaker": speaker,
                "segment_id": segment_id,
                "channel": segment.attrib.get("channel"),
                "start": segment.attrib.get("transcriber_start"),
                "end": segment.attrib.get("transcriber_end"),
                "word_ids": segment_word_ids,
                "text": segment_text
            })

        # Preserve words that have no segment assignment.
        for word_id in word_order:

            if word_id not in referenced_words:

                word = words[word_id]

                unsegmented_words.append({
                    "meeting_id": meeting_id,
                    "speaker": speaker,
                    "word_id": word_id,
                    "text": word["text"],
                    "start": word["start"],
                    "end": word["end"]
                })

    # Sort segments chronologically.
    def time_key(segment):

        try:
            start = float(segment["start"])
        except (TypeError, ValueError):
            start = float("inf")

        try:
            end = float(segment["end"])
        except (TypeError, ValueError):
            end = float("inf")

        return (start, end, segment["speaker"])

    structured_segments.sort(key=time_key)

    # Generate readable speaker-labelled transcript.
    readable_lines = []

    for segment in structured_segments:

        if segment["text"]:
            readable_lines.append(
                f'{segment["speaker"]}: {segment["text"]}'
            )

    readable_transcript = "\n".join(readable_lines)

    result = {
        "meeting_id": meeting_id,
        "segments": structured_segments,
        "unsegmented_words": unsegmented_words,
        "readable_transcript": readable_transcript
    }

    return result


# Test reconstruction on ES2002a.
result = reconstruct_meeting("ES2002a")

print("Meeting:", result["meeting_id"])
print("Reconstructed segments:", len(result["segments"]))
print("Unsegmented words:", len(result["unsegmented_words"]))
print("Readable transcript characters:",
      len(result["readable_transcript"]))

print("\n========== FIRST 10 STRUCTURED SEGMENTS ==========")

print(json.dumps(
    result["segments"][:10],
    indent=2,
    ensure_ascii=False
))

print("\n========== READABLE TRANSCRIPT (FIRST 30 LINES) ==========")

print("\n".join(
    result["readable_transcript"].splitlines()[:30]
))

In [ ]:
for meeting_id in ["ES2008a", "TS3003d"]:

    result = reconstruct_meeting(meeting_id)

    print(f"\n========== {meeting_id} ==========")
    print("Segments:", len(result["segments"]))
    print("Unsegmented words:", len(result["unsegmented_words"]))

    print("\nUnsegmented word details:")

    for word in result["unsegmented_words"]:
        print(word)

    print("\nFirst 10 transcript lines:")
    print("\n".join(
        result["readable_transcript"].splitlines()[:10]
    ))

## 4. Transcript cleaning

In [ ]:
### TRANSCRIPT CLEANUP FUNCION

import re

def clean_transcript(readable_transcript):

    # Remove um and uh, including commas or periods immediately after them
    transcript = re.sub(
        r"\b(?:um|uh)\b[,.]?",
        "",
        readable_transcript,
        flags=re.IGNORECASE)

    # Remove underscores wherever they appear
    transcript = transcript.replace("_", "")

    # Remove isolated single-character words from dialogue.
    # Preserve speaker labels A:, B:, C:, D: and standalone A/a.
    cleaned_lines = []

    for line in transcript.splitlines():
        match = re.match(r"^([A-D]:)(.*)$", line)

        if match:
            speaker, dialogue = match.groups()

            dialogue = re.sub(
                r"(?<![\w'])\b(?![Aa]\b)[A-Za-z]\b(?!')",
                "",
                dialogue)

            line = speaker + dialogue

        cleaned_lines.append(line)

    transcript = "\n".join(cleaned_lines)

    # Remove spaces before punctuation
    transcript = re.sub(r"\s+([,.?!])", r"\1", transcript)

    # Remove commas immediately after speaker labels
    transcript = re.sub(
        r"(?m)^([A-D]):\s*,\s*",
        r"\1: ", transcript)

    # Remove speaker lines containing only a period
    transcript = re.sub(
        r"(?m)^[A-D]:\s*\.\s*$\n?",
        "", transcript)

    # Remove empty dialogue lines
    transcript = re.sub(
        r"(?m)^[A-D]:\s*$\n?",
        "", transcript)

    # Remove commas immediately following periods
    transcript = re.sub(r"\.,+", ".", transcript)

    # Replace consecutive periods with a single period
    transcript = re.sub(r"\.{2,}", ".", transcript)

    # Replace consecutive commas with a single comma
    transcript = re.sub(r",{2,}", ",", transcript)

    # Remove extra spaces
    transcript = re.sub(r" {2,}", " ", transcript)

    return transcript

In [ ]:
# TESTING CLEAN TRANSCRIPT OUTPUT

result = reconstruct_meeting("ES2008a")

cleaned_transcript = clean_transcript(
    result["readable_transcript"]
)

print(cleaned_transcript)

## 5. Dataset construction and export

In [ ]:
# CONSTRUCT MEETING-LEVEL DATASET (stored in meeting_dataset)


from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

# Locate AMI abstractive annotation files
annotation_files = list(
    Path("ami_corpus").rglob("*.abssumm.xml")
)

print(f"Found annotation files: {len(annotation_files)}")

def get_tag_name(tag):
    """Remove XML namespace from a tag name."""
    return tag.split("}")[-1]

def extract_labels(xml_path):
    """Extract the four abstractive annotation categories."""
    root = ET.parse(xml_path).getroot()

    labels = {
        "abstract": "",
        "actions": "",
        "decisions": "",
        "problems": ""
    }

    for section in root:
        section_name = get_tag_name(section.tag)

        if section_name not in labels:
            continue

        sentences = []

        for element in section.iter():
            if get_tag_name(element.tag) == "sentence":
                sentence = "".join(element.itertext()).strip()

                if sentence:
                    sentences.append(sentence)

        labels[section_name] = " ".join(sentences)

    return labels


# Construct one record per meeting
meeting_dataset = []
failed_meetings = []

for annotation_path in annotation_files:

    meeting_id = annotation_path.name.split(".")[0]

    try:
        # Extract reference annotations
        labels = extract_labels(annotation_path)

        # Reconstruct and clean the transcript
        result = reconstruct_meeting(meeting_id)

        transcript = clean_transcript(
            result["readable_transcript"]
        )

        if not transcript.strip():
            raise ValueError("Empty reconstructed transcript")

        record = {
            "meeting_id": meeting_id,
            "transcript": transcript,
            **labels
        }

        meeting_dataset.append(record)

    except Exception as error:
        failed_meetings.append({
            "meeting_id": meeting_id,
            "error": str(error)
        })


# Dataset audit
print("\n" + "=" * 60)
print("MEETING-LEVEL DATASET AUDIT")
print("=" * 60)

print(f"Successfully constructed meetings: {len(meeting_dataset)}")
print(f"Failed meetings: {len(failed_meetings)}")

label_counts = Counter()

for label in ["abstract", "actions", "decisions", "problems"]:
    non_empty = sum(
        bool(record[label].strip())
        for record in meeting_dataset
    )

    label_counts[label] = non_empty

    print(
        f"{label.capitalize():12} "
        f"Non-empty: {non_empty:3} | "
        f"Empty: {len(meeting_dataset) - non_empty:3}"
    )

print("\nTotal non-empty task targets:", sum(label_counts.values()))

if failed_meetings:
    print("\nFailed meetings:")
    for item in failed_meetings:
        print(item)

# Inspect one constructed record
if meeting_dataset:
    sample = meeting_dataset[0]

    print("\n" + "=" * 60)
    print("SAMPLE RECORD")
    print("=" * 60)

    print("Meeting ID:", sample["meeting_id"])
    print("Transcript characters:", len(sample["transcript"]))

    for label in ["abstract", "actions", "decisions", "problems"]:
        print(f"\n{label.upper()} ({len(sample[label])} characters):")
        print(sample[label][:500])

In [ ]:
### CONSTRUCT TASK EXAMPLES


# Convert meeting-level records into task-specific examples

task_instructions = {
    "abstract": "summarize: ",
    "actions": "extract action items: ",
    "decisions": "extract decisions: ",
    "problems": "identify problems and unresolved issues: "
}

task_examples = []

for meeting in meeting_dataset:

    transcript = meeting["transcript"]

    for task, instruction in task_instructions.items():

        target = meeting[task].strip()

        # Skip empty annotation targets
        if not target:
            continue

        task_examples.append({
            "meeting_id": meeting["meeting_id"],
            "task": task,
            "input": instruction + transcript,
            "target": target
        })


# Audit task-specific examples
print("=" * 60)
print("TASK-SPECIFIC DATASET AUDIT")
print("=" * 60)

print("Total examples:", len(task_examples))

for task in task_instructions:

    count = sum(
        example["task"] == task
        for example in task_examples
    )

    print(f"{task.capitalize():12}: {count}")

# Inspect one example from each task
print("\n" + "=" * 60)
print("SAMPLE TASK EXAMPLES")
print("=" * 60)

for task in task_instructions:

    example = next(
        item for item in task_examples
        if item["task"] == task
    )

    print(f"\nTASK: {example['task']}")
    print("MEETING:", example["meeting_id"])
    print("INPUT PREFIX:", example["input"][:150])
    print("TARGET:", example["target"][:300])

In [ ]:
import pandas as pd

# Convert the meeting-level dataset into a DataFrame
df = pd.DataFrame(meeting_dataset)

# Arrange columns in a consistent order
df = df[
    ["meeting_id", "transcript", "abstract",
     "actions", "decisions", "problems"]
]

print("Dataset shape:", df.shape)

# Display the complete dataset
display(df)

# Spreadsheet-friendly format
df.to_csv("ami_meeting_dataset.csv", index=False)

# Structured format
df.to_json(
    "ami_meeting_dataset.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

import json

# Task-level examples (one row per meeting x task)
with open("ami_task_examples.jsonl", "w", encoding="utf-8") as f:
    for example in task_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Meeting-level records:", len(meeting_dataset))
print("Task-level examples  :", len(task_examples))
print("Saved: ami_meeting_dataset.csv, ami_meeting_dataset.jsonl, ami_task_examples.jsonl")